In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [17]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [18]:
len(words)

32033

In [21]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# print(stoi)
itos = {v: k for k, v in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [106]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        print(''.join(itos[i] for i in context),' --> ', ch)
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # crop and append
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
...  -->  e
..e  -->  m
.em  -->  m
emm  -->  a
mma  -->  .
olivia
...  -->  o
..o  -->  l
.ol  -->  i
oli  -->  v
liv  -->  i
ivi  -->  a
via  -->  .
ava
...  -->  a
..a  -->  v
.av  -->  a
ava  -->  .
isabella
...  -->  i
..i  -->  s
.is  -->  a
isa  -->  b
sab  -->  e
abe  -->  l
bel  -->  l
ell  -->  a
lla  -->  .
sophia
...  -->  s
..s  -->  o
.so  -->  p
sop  -->  h
oph  -->  i
phi  -->  a
hia  -->  .


In [105]:
X.shape, X.dtype, Y.shape, Y. dtype 

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [47]:
C = torch.randn((27, 2))

In [104]:
C[X].shape

torch.Size([32, 3, 2])

In [123]:
# example on how pytorch indexing works. C[X]
eg = torch.tensor([[0, 0, 1],[0, 1, 1]])
print('eg.shape:  ', eg.shape)
print('C[eg].shape', C[eg].shape)
C[eg]

eg.shape:   torch.Size([2, 3])
C[eg].shape torch.Size([2, 3, 2])


tensor([[[-0.6108,  0.0158],
         [-0.6108,  0.0158],
         [-1.2004,  2.6672]],

        [[-0.6108,  0.0158],
         [-1.2004,  2.6672],
         [-1.2004,  2.6672]]])

In [125]:
C[X][13, 2]

tensor([-1.2004,  2.6672])

In [126]:
X[13, 2]

tensor(1)

In [131]:
# C[X][13, 2] = C[X[13, 2]]
C[1]

tensor([-1.2004,  2.6672])

In [241]:
# embedd all of the integers in X
emb =  C[X]
emb.shape

torch.Size([32, 3, 2])

In [242]:
# implementing the hidden layer

# no. of inputs to the hidden layer is 6 with our current settings. Why?
# because we have 2 dimensional embeddings of 3 units of look-up tables.(3 previous chars(context_length) to predict the 4th char)
W1 = torch.randn((6, 100))
b1 = torch.rand(100)

emb @ W1 + b1  # work won't 

RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x2 and 6x100)

In [211]:
# We need to transform emb to (32, 6) from (32, 3, 2) to perform dot product with W1 (6, 100 )

# emb[:, 0, :]
emb[:, 0, :].shape
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1 )

torch.Size([32, 2])

In [206]:
a = torch.tensor([[[1, 2], [3, 4], [5, 6]],[[7, 8], [9, 10], [11, 12]], [[13, 14], [15, 16], [17, 18]]])
print(a)
print(a.shape)
# print(torch.unbind(a, 1))
print(torch.cat(torch.unbind(a, 1), 1))
print(torch.cat(torch.unbind(a, 1), 1).shape)

# torch.cat(torch.unbind(emb, 1), 1) == torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1 )

tensor([[[ 1,  2],
         [ 3,  4],
         [ 5,  6]],

        [[ 7,  8],
         [ 9, 10],
         [11, 12]],

        [[13, 14],
         [15, 16],
         [17, 18]]])
torch.Size([3, 3, 2])
tensor([[ 1,  2,  3,  4,  5,  6],
        [ 7,  8,  9, 10, 11, 12],
        [13, 14, 15, 16, 17, 18]])
torch.Size([3, 6])


In [216]:
# this is one way to transform the input to (32, 6) tensor using torch.cat(torch.unbind(emb,1),1)
# but turns out there is a different way which is a much efficient way than this! (explained in next cell)

# torch.unbind(emb, 1)
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [227]:
# calling .view() is extremely effiecient in pytorch. Reason for that is that in each tensor, 
# there's something called the underlying storage.

# tensor.view() works as long as the product of the dims is equal to the size of the tensor. 
a = torch.arange(18)
print(a)
a.view(2, 9)
# a.view(9, 2)
# a.view(3, 3, 2)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [229]:
# the storage is just the numbers always stored as a one-dimensional vector and 
# this is how a tensor is represented in the computer memory.
# so when view() is manipulating how this 1-d sequence is interpreted as a n-dimensional tensor.
# No memory is changed, copied, moved or deleted while we call tensor.view(), 
# instead some storage attributes storage offsets, strides and shapes are amnipulated.
# For more information:  (https://blog.ezyang.com/2019/05/pytorch-internals/)

In [244]:
emb.shape

torch.Size([32, 3, 2])

In [255]:
# emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)
# emb.view(-1, 6) # pytorch will infer the value at -1 automatically!

h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # output of the hidden layer
h

tensor([[ 0.5936,  0.9649,  0.8550,  ...,  0.1879,  0.4853,  0.7681],
        [-0.6459,  0.9997, -0.9334,  ...,  0.9999, -0.6459,  0.9930],
        [ 0.7898,  0.9720,  0.9566,  ..., -1.0000,  0.9988, -0.3812],
        ...,
        [ 0.3982,  0.9999, -0.9219,  ..., -0.9831,  0.1848, -0.9977],
        [ 0.0919,  1.0000,  0.9948,  ...,  0.9335,  0.9753, -0.9881],
        [ 0.5563,  0.9631,  0.9999,  ..., -0.8196,  0.8688, -0.9842]])

In [246]:
h.shape

torch.Size([32, 100])

In [247]:
(emb.view(-1, 6) @ W1).shape

torch.Size([32, 100])

In [249]:
b1.shape

torch.Size([100])

In [252]:
# 32, 100   (emb.view(-1, 6) @ W1).shape
# 1,  100    updated b1.shape from broadcasting

# broadcasting process, step by step.
# step 1: Align to the right when there is shape mismatch
# step 2: Create a fake dimension(1) to the left. -> (1, 100)
# step 3: Copy the (1, 100) row vector n times to match the other tensor(n=32 in this case)
# step 4: Complete element-wise addition:  (emb.view(-1, 6) @ W1) + b1